# ATS scoring engine — pipeline stages, one at a time

Each stage of the analysis pipeline is a plain function or a small adapter, so it can be run here on
sample data **without starting the FastAPI app** (Req 13). Run the cells top to bottom.

| Stage | Code | Needs |
|---|---|---|
| 1. Retrieve file | `LocalPathFileStore.read` | the sample file |
| 2. Extract text | `extract_text` | — |
| 3. Prepare input | `prepare_input` | — |
| 4. Build prompt | `build_prompt` | — |
| 5. Run LLM | `OllamaLlmClient.complete` | Ollama running with `qwen2.5:3b` |
| 6. Parse result | `parse_result` | — |

Callback delivery and the full pipeline are covered by the unit tests (`tests/unit/test_deliver.py`,
`tests/unit/test_run.py`), not here.


In [1]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))
ASSETS = PROJECT / "tests" / "assets"
print("project:", PROJECT)

project: C:\Users\mdtar\source\repos\ATS\ats-fastapi


## 1. Retrieve the CV file

The reference is an absolute path; the type comes from the file's content, not its name.

In [2]:
from app.adapters.local_path_file_store import LocalPathFileStore

store = LocalPathFileStore()
data, file_type = await store.read(str(ASSETS / "sample_cv.pdf"))
print(file_type, len(data), "bytes")

pdf 61615 bytes


## 2. Extract text

Every page is read. Try `sample_cv.docx`, `sample_cv_3_pages.pdf`, or `blank.pdf` (raises `ExtractionEmptyError`).

In [3]:
from app.pipeline.extract import extract_text

cv_text = extract_text(data, file_type)
print(cv_text)

Md Tariqul Alam
Software Engineer | .NET, AWS and Generative AI
Toronto, ON, Canada | +1 (382) 880-5317 | mdtariqul.alam@hotmail.com
linkedin.com/in/md-tariqul-alam-70284a120 | github.com/tariqul2814
Work Authorization: Open Work Permit (Canada)
SUMMARY
Software Engineer with 7+ years in .NET, Python, and AWS. Delivers full-stack solutions in Agile/Scrum teams, modernizes legacy
.NET platforms, and builds RAG and agentic AI on Amazon Bedrock; strong in ASP.NET Core, Blazor, SQL Server,
Docker/Kubernetes, Terraform, and CI/CD.
PROFESSIONAL EXPERIENCE
IQVIA Dhaka, Bangladesh
Software Developer Feb 2022 – Sep 2026
• Built SaaS-ERP reporting features for pharma clients (Pfizer, Technimede) in C#, Blazor, SQL Server, and JavaScript.
• Hardened the architecture by resolving defects, tuning performance, and adding OAuth 2.0 authentication via Salesforce.
• Established CI/CD pipelines in Azure DevOps and automated monitoring with health checks and Swagger (OpenAPI) docs.
• Delivered features i

## 3. Prepare input

Normalises, neutralises prompt delimiters, and applies the character budget (whole / truncate / reject).

In [4]:
from app.pipeline.prepare import InputLimits, prepare_input

limits = InputLimits(max_cv_chars=24000, max_jd_chars=8000, hard_ceiling_chars=120000)
job_description = (ASSETS / "sample_jd.txt").read_text(encoding="utf-8")
prepared = prepare_input(cv_text, job_description, limits)
print("truncated:", prepared.truncated, "| CV chars:", prepared.original_chars)

# The same CV against a tiny budget, to see the head + tail truncation:
tiny = prepare_input(cv_text, job_description, InputLimits(250, 8000, 120000))
print(tiny.cv_text)

truncated: False | CV chars: 4124
Md Tariqul Alam
Software Engineer | .NET, AWS and Generative AI
Toronto, ON, Canada | +1 (382) 880-5317 | mdtariqul.alam@hotmail.com
[… truncated …]
Engineering


## 4. Build the prompt

Loads the template the service is configured to use (`APP_PROMPT__TEMPLATE`, default
`ats_rubric_v1`) rather than a fixed name, prints the active template (name + unfilled system/user),
then fills its `{{CV_TEXT}}` / `{{JOB_DESCRIPTION}}` placeholders from the prepared input.

In [5]:
from app.config import PromptSettings, Settings
from app.pipeline.prompt import build_prompt, load_template

# The template the running service would use (APP_PROMPT__TEMPLATE, default ats_rubric_v1).
# Settings reads the project's .env regardless of this notebook's working directory; if the service
# config is incomplete on this checkout (e.g. no webhook secret), fall back to the packaged default.
try:
    template_name = Settings(_env_file=str(PROJECT / ".env")).prompt.template
except Exception as exc:  # noqa: BLE001 — notebook: full service config is optional for this stage
    print(f"(could not read service settings, using default template: {exc})")
    template_name = PromptSettings().template

template = load_template(template_name)
print(f"active prompt template: {template.name}\n")
print("=== SYSTEM ===")
print(template.system[:600], "...\n")
print("=== USER (placeholders unfilled) ===")
print(template.user, "\n")

# Fill the template with the prepared inputs to get the messages the model is actually sent.
messages = build_prompt(prepared, template)
print("=== FILLED USER MESSAGE ===")
print(messages[1]["content"])

active prompt template: ats_rubric_v3

=== SYSTEM ===
You are an expert ATS (Applicant Tracking System) analyst and senior technical recruiter. You assess how well a candidate's CV matches a specific job description and how well the CV will survive automated ATS screening.

## Ground rules
- Use ONLY what is written in the CV. Never invent skills, employers, dates, numbers, or achievements.
- When you suggest a change, quote the CV's actual text in "original" and keep "suggested" truthful to it.
- Count synonyms and close variants as matches (e.g. "JS" = "JavaScript", "Postgres" = "PostgreSQL", "k8s" = "Kubernetes").
- Score strictly and realisti ...

=== USER (placeholders unfilled) ===
Assess the CV below against the job description.

<job_description>
{{JOB_DESCRIPTION}}
</job_description>

<candidate_cv>
{{CV_TEXT}}
</candidate_cv>

Return the JSON object now. 

=== FILLED USER MESSAGE ===
Assess the CV below against the job description.

<job_description>
We're looking for Softwar

## 5. Run the LLM

Needs Ollama running locally with the model pulled (`ollama pull qwen2.5:3b`). On a CPU-only machine this can take a few minutes. Set `RUN_LLM = False` to skip.

In [6]:
from app.adapters.ollama_llm_client import OllamaLlmClient
from app.config import OllamaSettings

RUN_LLM = True
llm = OllamaLlmClient(OllamaSettings(), timeout_seconds=480)
print("Ollama ready:", await llm.check_ready())
raw = None
if RUN_LLM:
    import time

    started = time.perf_counter()
    try:
        raw = await llm.complete(messages)
        print(f"model replied in {time.perf_counter() - started:.0f}s, {len(raw)} chars\n")
        print(raw)
    except Exception as exc:  # noqa: BLE001 — debug cell: show any failure, don't raise
        print("LLM call failed:", type(exc).__name__, exc)

Ollama ready: True
model replied in 239s, 5076 chars

{
  "score": 55,
  "band": "Moderate",
  "breakdown": [
    {
      "category": "keyword_match",
      "score": 13,
      "max": 35,
      "reason": "The CV contains relevant skills and technologies mentioned in the job description, but not all are explicitly mentioned."
    },
    {
      "category": "experience_relevance",
      "score": 12,
      "max": 25,
      "reason": "The candidate has relevant experience in full-stack development, but the roles do not match the job description's specific requirements."
    },
    {
      "category": "achievements_impact",
      "score": 7,
      "max": 15,
      "reason": "The candidate lists achievements but lacks specific quantifiable results or impact related to the job's requirements."
    },
    {
      "category": "ats_formatting",
      "score": 13,
      "max": 15,
      "reason": "The CV has standard section headings and a plain structure, making it parseable by ATS."
    },
    {

## 6. Parse and validate

Turns the model's reply from stage 5 into the validated result and renders the report. The saved sample replies show the repairs (clamped scores, recomputed total/band) and the failure case.

In [7]:
from app.pipeline.errors import ResultParseError
from app.pipeline.parse import parse_result

fixed = parse_result((ASSETS / "model_output_out_of_range.json").read_text(encoding="utf-8"))
print("repaired:", fixed.score, fixed.band, [(c.category, c.score) for c in fixed.breakdown])
try:
    parse_result((ASSETS / "model_output_malformed.txt").read_text(encoding="utf-8"))
except ResultParseError as exc:
    print("malformed ->", exc.reason, "|", exc)

source = "model reply from stage 5" if raw else "saved sample reply (stage 5 skipped or failed)"
result = parse_result(raw) if raw else parse_result((ASSETS / "model_output_valid.json").read_text(encoding="utf-8"))
print(f"\n{source} -> score {result.score} ({result.band})\n")
print(result.reportMarkdown)

repaired: 70 Good [('keyword_match', 35), ('experience_relevance', 0), ('achievements_impact', 12), ('ats_formatting', 15), ('education_certifications', 8)]
malformed -> result_unparseable | no JSON object in model output

model reply from stage 5 -> score 48 (Weak)

# ATS assessment

**Score:** 48/100 — **Weak**

The CV shows relevant skills and experience, but the roles do not match the job description's specific requirements. The candidate lacks specific achievements and quantifiable results.

## Breakdown

| Category | Score | Reason |
|---|---|---|
| Keyword match | 13/35 | The CV contains relevant skills and technologies mentioned in the job description, but not all are explicitly mentioned. |
| Experience relevance | 12/25 | The candidate has relevant experience in full-stack development, but the roles do not match the job description's specific requirements. |
| Achievements impact | 7/15 | The candidate lists achievements but lacks specific quantifiable results or impact relat